In [2]:
# ============================================================
# Kaggle setup — automatically locate the attached project dataset
# ============================================================

from pathlib import Path
import os

KAGGLE_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/rdd2022_reproduction")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

REQUIRED_ITEMS = [
    "common.py",
    "frozen_split",
    "test_dark_mild",
    "test_dark_severe",
    "runs",
]

def find_project_root(search_root=KAGGLE_INPUT):
    """Find the Kaggle Input directory containing the saved project artifacts."""
    search_root = Path(search_root)

    if not search_root.exists():
        raise FileNotFoundError(
            "Kaggle input directory was not found. "
            "Attach the project dataset using the Kaggle notebook Input panel."
        )

    # First check the input root itself.
    candidates = [search_root]

    # Then inspect directories under /kaggle/input.
    candidates.extend(p for p in search_root.rglob("*") if p.is_dir())

    for candidate in candidates:
        if all((candidate / item).exists() for item in REQUIRED_ITEMS):
            return candidate.resolve()

    # Helpful diagnostic if the dataset was attached under an unexpected structure.
    top_level = sorted(str(p.relative_to(search_root)) for p in search_root.iterdir())
    raise FileNotFoundError(
        "Could not locate the saved rdd2022_project artifacts under /kaggle/input.\n\n"
        "The attached Kaggle Dataset must contain these items together:\n"
        + "\n".join(f"  - {x}" for x in REQUIRED_ITEMS)
        + "\n\nCurrent /kaggle/input contents:\n"
        + "\n".join(f"  - {x}" for x in top_level[:100])
    )

REPO_ROOT = find_project_root()

print("Kaggle input project root:")
print(" ", REPO_ROOT)
print()
print("Writable working directory:")
print(" ", WORK_ROOT)

missing = [x for x in REQUIRED_ITEMS if not (REPO_ROOT / x).exists()]
if missing:
    raise FileNotFoundError(f"Missing required project artifacts: {missing}")

print("\n✅ Project dataset found")
print("✅ No Google Drive is required")
print("✅ No retraining will be performed")


Kaggle input project root:
  /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project

Writable working directory:
  /kaggle/working/rdd2022_reproduction

✅ Project dataset found
✅ No Google Drive is required
✅ No retraining will be performed


# RDD2022 — YOLOv11n-CDL Low-Light Robustness
## Reproduce / Validate Saved Results on Kaggle (CPU-only, No Retraining)

This notebook is the **reproduction/validation notebook** for the project.

It loads the saved R1–R5 checkpoints and the frozen clean/dark test data from an attached **Kaggle Dataset**, then evaluates them locally on Kaggle.

### Important
- **No model is retrained.**
- The frozen split is not regenerated.
- The saved dark test sets are not regenerated.
- The saved `best.pt` checkpoints are loaded directly.
- Evaluation outputs are written only to `/kaggle/working`.
- The Kaggle Input dataset is read-only; the notebook therefore copies evaluation datasets to the writable working directory when necessary.

### Expected Kaggle Dataset contents

The attached Dataset should contain the saved project folder/artifacts, including:

```text
common.py
frozen_split/
    rdd2022_yolo_4k.zip
test_dark_mild/
test_dark_severe/
runs/
    R1_baseline_scratch/weights/best.pt
    R2_cdl_scratch/weights/best.pt
    R3_cdl_aug/weights/best.pt
    R4_cdl_age_aug_v2/weights/best.pt
    R5_cdl_gated_age_aug/weights/best.pt
main_results_table.csv
results_tables.csv
...
```

The notebook automatically searches `/kaggle/input` for the directory containing these artifacts, so you do **not** need to hard-code the Kaggle Dataset name.


### Cleaned notebook version
Duplicate setup/checkpoint/comparison cells have been removed. Run the remaining cells from top to bottom after restarting the session.

## 1. Repository setup

In [3]:
# ============================================================
# Prepare writable local copies of dark test sets
# ============================================================

from pathlib import Path
import shutil
import yaml

DARK_MILD_SRC = REPO_ROOT / "test_dark_mild"
DARK_SEVERE_SRC = REPO_ROOT / "test_dark_severe"

DARK_MILD_LOCAL = WORK_ROOT / "test_dark_mild"
DARK_SEVERE_LOCAL = WORK_ROOT / "test_dark_severe"

def prepare_dark_dataset(src, dst, yaml_name):
    """
    Copy the saved dark test dataset to Kaggle's writable storage.

    The Kaggle Input dataset is read-only, while Ultralytics may create
    label caches during validation. Keeping a local copy prevents any
    attempt to modify the attached dataset.
    """

    src = Path(src)
    dst = Path(dst)

    if not src.exists():
        raise FileNotFoundError(f"Dark dataset not found: {src}")

    if not ((dst / "images").exists() and (dst / "labels").exists()):
        if dst.exists():
            shutil.rmtree(dst)

        print(f"Copying {src} -> {dst}")
        shutil.copytree(src, dst)

    # Remove stale cache files from previous runs.
    for cache_file in dst.rglob("*.cache"):
        try:
            cache_file.unlink()
        except Exception:
            pass

    # Ultralytics requires train + val + test keys.
    # This is evaluation-only, so all three point to the same images.
    yaml_path = WORK_ROOT / yaml_name

    yaml_content = {
        "path": str(dst),
        "train": "images",
        "val": "images",
        "test": "images",
        "names": {
            0: "D00",
            1: "D10",
            2: "D20",
            3: "D40",
        },
    }

    with open(yaml_path, "w") as f:
        yaml.safe_dump(yaml_content, f, sort_keys=False)

    print(f"Created: {yaml_path}")
    return yaml_path

MILD_YAML = prepare_dark_dataset(
    DARK_MILD_SRC,
    DARK_MILD_LOCAL,
    "data_dark_mild.yaml",
)

SEVERE_YAML = prepare_dark_dataset(
    DARK_SEVERE_SRC,
    DARK_SEVERE_LOCAL,
    "data_dark_severe.yaml",
)

print("\nDark evaluation datasets ready.")
print("MILD  :", MILD_YAML)
print("SEVERE:", SEVERE_YAML)


Created: /kaggle/working/rdd2022_reproduction/data_dark_mild.yaml
Created: /kaggle/working/rdd2022_reproduction/data_dark_severe.yaml

Dark evaluation datasets ready.
MILD  : /kaggle/working/rdd2022_reproduction/data_dark_mild.yaml
SEVERE: /kaggle/working/rdd2022_reproduction/data_dark_severe.yaml


## Evaluation of saved checkpoints (no retraining)

This section loads the saved R1–R5 checkpoints and evaluates them on the frozen clean, mild-dark, and severe-dark test sets. No `train()` call is made. The dark evaluation datasets are local writable copies so Ultralytics can create label caches without modifying the shared Kaggle Dataset artifacts. The dataset YAMLs define `train`, `val`, and `test` keys because Ultralytics expects those keys for detection datasets; all three point to the evaluation image folder here. citeturn0search0turn0search2


In [4]:
# Install the versions used by the experiment.
# This cell installs dependencies only; it does NOT train or evaluate anything.

%pip install -q "ultralytics==8.3.152" "albumentations==1.4.18" --no-deps
%pip install -q "albucore==0.0.17" "eval-type-backport"


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import torchvision
import ultralytics
import albumentations

print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("albumentations:", albumentations.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Evaluation can still run, but it will be slower.")


ultralytics: 8.3.152
torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
albumentations: 1.4.18
CUDA available: True
GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


## 2. Load the saved project artifacts

The project artifacts are read from the attached Kaggle Dataset.

The attached Dataset is treated as **read-only**. Only temporary extracted data, copied dark test sets, YAML files, and validation outputs are written under `/kaggle/working/rdd2022_reproduction`.


In [6]:

from pathlib import Path

FROZEN_DIR = REPO_ROOT / "frozen_split"
FROZEN_ZIP = FROZEN_DIR / "rdd2022_yolo_4k.zip"
EXTRACTED_DATA = FROZEN_DIR / "rdd2022_yolo_4k"

DARK_MILD_DIR = REPO_ROOT / "test_dark_mild"
DARK_SEVERE_DIR = REPO_ROOT / "test_dark_severe"
RUNS_DIR = REPO_ROOT / "runs"
FIGURES_DIR = REPO_ROOT / "figures"

required = {
    "frozen_split": FROZEN_DIR,
    "extracted dataset": EXTRACTED_DATA,
    "test_dark_mild": DARK_MILD_DIR,
    "test_dark_severe": DARK_SEVERE_DIR,
    "runs": RUNS_DIR,
}

for name, path in required.items():
    print(f"{name:20s}:", "OK" if path.exists() else "MISSING")

missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required project artifacts: " + ", ".join(missing)
    )

# Use the extracted dataset directly.
# No ZIP extraction or retraining is needed.
YOLO_DIR = EXTRACTED_DATA

print("\n✅ Frozen dataset found.")
print("Dataset path:", YOLO_DIR)


frozen_split        : OK
extracted dataset   : OK
test_dark_mild      : OK
test_dark_severe    : OK
runs                : OK

✅ Frozen dataset found.
Dataset path: /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/frozen_split/rdd2022_yolo_4k


In [7]:

# Use the already-extracted frozen dataset.
# Do NOT try to open or extract the missing ZIP file.

from pathlib import Path

YOLO_DIR = REPO_ROOT / "frozen_split" / "rdd2022_yolo_4k"

print("Checking extracted dataset:", YOLO_DIR)

if not YOLO_DIR.exists():
    raise FileNotFoundError(
        f"Extracted dataset folder not found: {YOLO_DIR}"
    )

# Verify the expected test folders exist.
for subpath in [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test",
]:
    path = YOLO_DIR / subpath
    print(f"{subpath:15s}:", "OK" if path.exists() else "MISSING")

if not (YOLO_DIR / "images" / "test").exists():
    raise FileNotFoundError(
        "The extracted dataset does not contain images/test. "
        "Check the dataset folder structure before continuing."
    )

test_images = list((YOLO_DIR / "images" / "test").glob("*"))

print("\nYOLO data root:", YOLO_DIR)
print("Test image files:", len(test_images))


Checking extracted dataset: /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/frozen_split/rdd2022_yolo_4k
images/train   : OK
images/val     : OK
images/test    : OK
labels/train   : OK
labels/val     : OK
labels/test    : OK

YOLO data root: /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/frozen_split/rdd2022_yolo_4k
Test image files: 403


In [8]:
import yaml

DATA_DIR = WORK_ROOT / "yaml"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_YAML = DATA_DIR / "rdd2022_clean.yaml"
CLEAN_YAML.write_text(f'''path: {YOLO_DIR.as_posix()}
train: images/train
val: images/val
test: images/test
names:
  0: D00
  1: D10
  2: D20
  3: D40
''')

print("Clean YAML:", CLEAN_YAML)


Clean YAML: /kaggle/working/rdd2022_reproduction/yaml/rdd2022_clean.yaml


## 3. Register the project's custom modules

The saved R2–R5 checkpoints depend on the project's custom `ConvSmart`, `DSA`, and AGE modules.

`common.py` is loaded directly from the attached project Dataset.


In [9]:
# The saved checkpoints for R2-R5 use project-specific modules.
# common.py is the tracked source of truth for ConvSmart, DSA and utilities.

import sys
import importlib

sys.path.insert(0, str(REPO_ROOT))

import common
importlib.reload(common)

print("Loaded:", REPO_ROOT / "common.py")
print("Available registration helper:", hasattr(common, "register_custom_modules"))

if hasattr(common, "register_custom_modules"):
    common.register_custom_modules()
    print("Custom modules registered.")


Loaded: /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/common.py
Available registration helper: True
Registered ConvSmart and DSA into ultralytics.nn.tasks.parse_model (ultralytics 8.3.152).
Custom modules registered.



### AGE compatibility

R4 and R5 both use a module named `AGE`, but they use different forward behaviour:

- **R4:** naive AGE, `gamma = 0.5 + sigmoid(...)`
- **R5:** brightness-gated AGE, where the correction is blended toward identity according to image brightness.

The reproduction notebook registers the appropriate implementation **before loading each checkpoint** so that the saved R4 and R5 models are evaluated with the same AGE definitions used by the experiment.


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import ultralytics.nn.tasks as tasks
import __main__

class NaiveAGE(nn.Module):
    def __init__(self):
        super().__init__()
        self.gamma_net = nn.Sequential(
            nn.Conv2d(3, 16, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(16, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
        )
        self.fc1 = nn.Linear(2048, 64)
        self.fc2 = nn.Linear(64, 1)
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x_small = F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)
        gamma = self.gamma_net(x_small).flatten(1)
        gamma = F.leaky_relu(self.fc1(gamma))
        gamma = self.fc2(gamma)
        gamma = 0.5 + torch.sigmoid(gamma)
        gamma = gamma.view(-1, 1, 1, 1)
        x = torch.clamp(x, 0, 1)
        return x ** gamma

class GatedAGE(nn.Module):
    def __init__(self, brightness_thresh=0.4):
        super().__init__()
        self.brightness_thresh = brightness_thresh
        self.gamma_net = nn.Sequential(
            nn.Conv2d(3, 16, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(16, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
        )
        self.fc1 = nn.Linear(2048, 64)
        self.fc2 = nn.Linear(64, 1)
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x_small = F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)
        with torch.no_grad():
            brightness = x_small.mean(dim=(1, 2, 3), keepdim=True)
            darkness_gate = torch.clamp(1 - brightness / self.brightness_thresh, 0, 1)
        gamma = self.gamma_net(x_small).flatten(1)
        gamma = F.leaky_relu(self.fc1(gamma))
        gamma = self.fc2(gamma)
        gamma_raw = 0.5 + torch.sigmoid(gamma)
        gamma_raw = gamma_raw.view(-1, 1, 1, 1)
        gamma = 1.0 + darkness_gate * (gamma_raw - 1.0)
        x = torch.clamp(x, 0, 1)
        return x ** gamma

def register_age(age_class):
    tasks.AGE = age_class
    ns = getattr(tasks, '_cdl_patched_ns', None)
    if ns is not None:
        ns['AGE'] = age_class
    __main__.AGE = age_class
    print('Registered AGE as:', age_class.__name__)


## 4. Locate the saved checkpoints

The notebook expects the five saved `best.pt` checkpoints from the completed experiment.


## 5. Load checkpoints — no training

The following cell only deserializes the saved checkpoints.

If a checkpoint is missing, the notebook stops instead of retraining anything.


In [12]:

from pathlib import Path

# Map each experiment to its saved best checkpoint.
CHECKPOINTS = {
    "R1_baseline_scratch":
        REPO_ROOT / "runs/R1_baseline_scratch/weights/best.pt",

    "R2_cdl_scratch":
        REPO_ROOT / "runs/R2_cdl_scratch/weights/best.pt",

    "R3_cdl_aug":
        REPO_ROOT / "runs/R3_cdl_aug/weights/best.pt",

    "R4_cdl_age_aug":
        REPO_ROOT / "runs/R4_cdl_age_aug_v2/weights/best.pt",

    "R5_cdl_gated_age_aug":
        REPO_ROOT / "runs/R5_cdl_gated_age_aug/weights/best.pt",
}

# Verify every checkpoint exists before loading
print("Checking saved checkpoints:\n")

for name, path in CHECKPOINTS.items():
    status = "OK" if path.is_file() else "MISSING"
    print(f"{name}: {status}")
    print(f"  {path}")

missing = [
    name for name, path in CHECKPOINTS.items()
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        f"Missing checkpoints for: {missing}"
    )

print("\nAll five best checkpoints are available.")


Checking saved checkpoints:

R1_baseline_scratch: OK
  /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/runs/R1_baseline_scratch/weights/best.pt
R2_cdl_scratch: OK
  /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/runs/R2_cdl_scratch/weights/best.pt
R3_cdl_aug: OK
  /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/runs/R3_cdl_aug/weights/best.pt
R4_cdl_age_aug: OK
  /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/runs/R4_cdl_age_aug_v2/weights/best.pt
R5_cdl_gated_age_aug: OK
  /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/runs/R5_cdl_gated_age_aug/weights/best.pt

All five best checkpoints are available.


In [13]:

from ultralytics import YOLO
import __main__

# Model names used in the saved-results reproduction experiment
RUN_NAMES = [
    "R1_baseline_scratch",
    "R2_cdl_scratch",
    "R3_cdl_aug",
    "R4_cdl_age_aug",
    "R5_cdl_gated_age_aug",
]

def load_saved_model(model_name):
    # R4 was serialized with the custom class name AGE.
    if model_name == "R4_cdl_age_aug":
        register_age(NaiveAGE)
        __main__.AGE = NaiveAGE
        print("R4: registered AGE -> NaiveAGE")

    elif model_name == "R5_cdl_gated_age_aug":
        register_age(GatedAGE)
        __main__.AGE = GatedAGE
        __main__.GatedAGE = GatedAGE
        print("R5: registered AGE -> GatedAGE")

    print(f"Loading saved checkpoint for {model_name} ...")

    model = YOLO(str(CHECKPOINTS[model_name]))

    print(f"Loaded {model_name} successfully.")
    return model


models = {}

for name in RUN_NAMES:
    models[name] = load_saved_model(name)

print("\n========================================")
print("Saved models loaded successfully")
print("========================================")
print("Loaded models:", list(models.keys()))


Loading saved checkpoint for R1_baseline_scratch ...
Loaded R1_baseline_scratch successfully.
Loading saved checkpoint for R2_cdl_scratch ...
Loaded R2_cdl_scratch successfully.
Loading saved checkpoint for R3_cdl_aug ...
Loaded R3_cdl_aug successfully.
Registered AGE as: NaiveAGE
R4: registered AGE -> NaiveAGE
Loading saved checkpoint for R4_cdl_age_aug ...
Loaded R4_cdl_age_aug successfully.
Registered AGE as: GatedAGE
R5: registered AGE -> GatedAGE
Loading saved checkpoint for R5_cdl_gated_age_aug ...
Loaded R5_cdl_gated_age_aug successfully.

Saved models loaded successfully
Loaded models: ['R1_baseline_scratch', 'R2_cdl_scratch', 'R3_cdl_aug', 'R4_cdl_age_aug', 'R5_cdl_gated_age_aug']


## 6. Reproduce clean / mild-dark / severe-dark evaluation

The evaluation uses:
- the frozen clean test split extracted from `rdd2022_yolo_4k.zip`,
- the saved `test_dark_mild` dataset,
- the saved `test_dark_severe` dataset.

No test data are regenerated.


In [14]:
# ============================================================
# Evaluate all saved checkpoints — NO RETRAINING
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

VAL_OUTPUT_ROOT = WORK_ROOT / "reproduction_val_outputs"
VAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Safety check: all evaluation YAMLs must already exist.
for _name, _path in {
    "test_clean": CLEAN_YAML,
    "test_dark_mild": MILD_YAML,
    "test_dark_severe": SEVERE_YAML,
}.items():
    if not Path(_path).exists():
        raise FileNotFoundError(
            f"{_name} YAML was not created: {_path}. "
            "Run the notebook from the top in order."
        )

EVAL_SETS = {
    "test_clean": CLEAN_YAML,
    "test_dark_mild": MILD_YAML,
    "test_dark_severe": SEVERE_YAML,
}

def evaluate_model(model, model_name, data_yaml, test_name):
    print(f"\nRunning evaluation: {model_name} | {test_name}")

    res = model.val(
        data=str(data_yaml),
        split="test",
        verbose=False,
        project=str(VAL_OUTPUT_ROOT / model_name),
        name=test_name,
        exist_ok=True,
        save=False,
        plots=False,
    )

    ap = res.box.ap50
    params_m = sum(p.numel() for p in model.model.parameters()) / 1e6

    row = {
        "model": model_name,
        "precision": float(res.box.mp),
        "recall": float(res.box.mr),
        "mAP50": float(res.box.map50),
        "mAP50-95": float(res.box.map),
        "AP50_D00": float(ap[0]),
        "AP50_D10": float(ap[1]),
        "AP50_D20": float(ap[2]),
        "AP50_D40": float(ap[3]),
        "params_M": round(params_m, 3),
        "test_set": test_name,
    }

    print(row)
    return row

rows = []

for model_name, model in models.items():
    for test_name, yaml_path in EVAL_SETS.items():
        print(f"\n{'=' * 10} {model_name} | {test_name} {'=' * 10}")
        rows.append(
            evaluate_model(
                model=model,
                model_name=model_name,
                data_yaml=yaml_path,
                test_name=test_name,
            )
        )

reproduced_df = pd.DataFrame(rows)

model_order = list(RUN_NAMES)
test_order = list(EVAL_SETS.keys())

reproduced_df["model"] = pd.Categorical(
    reproduced_df["model"],
    categories=model_order,
    ordered=True,
)

reproduced_df["test_set"] = pd.Categorical(
    reproduced_df["test_set"],
    categories=test_order,
    ordered=True,
)

reproduced_df = (
    reproduced_df
    .sort_values(["model", "test_set"])
    .reset_index(drop=True)
)

print("\n" + "=" * 60)
print("REPRODUCED RESULTS")
print("=" * 60)

display(reproduced_df)



========== R1_baseline_scratch | test_clean ==========

Running evaluation: R1_baseline_scratch | test_clean
Ultralytics 8.3.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients
val: Fast image access ✅ (ping: 0.2±0.3 ms, read: 128.2±72.0 MB/s, size: 80.0 KB)


val: Scanning /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/frozen_split/rdd2022_yolo_4k/labels/test... 403 images, 0 backgrounds, 0 corrupt: 100%|██████████| 403/403 [00:00<00:00, 779.59it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/frozen_split/rdd2022_yolo_4k/labels is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 26/26 [00:03<00:00,  7.39it/s]


                   all        403        955      0.424      0.309      0.274      0.115
Speed: 0.2ms preprocess, 3.8ms inference, 0.0ms loss, 1.3ms postprocess per image
{'model': 'R1_baseline_scratch', 'precision': 0.4240754684198452, 'recall': 0.30935106666883, 'mAP50': 0.2743394820852122, 'mAP50-95': 0.11533727086512575, 'AP50_D00': 0.30561878662626946, 'AP50_D10': 0.2288828887464712, 'AP50_D20': 0.40401404416931747, 'AP50_D40': 0.15884220879879063, 'params_M': 2.583, 'test_set': 'test_clean'}

========== R1_baseline_scratch | test_dark_mild ==========

Running evaluation: R1_baseline_scratch | test_dark_mild
Ultralytics 8.3.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2128.6±860.9 MB/s, size: 68.1 KB)


...

val: Scanning /kaggle/working/rdd2022_reproduction/test_dark_mild/labels.cache... 403 images, 0 backgrounds, 0 corrupt: 100%|██████████| 403/403 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 26/26 [00:07<00:00,  3.37it/s]

                   all        403        955      0.359      0.187      0.156     0.0647
Speed: 0.2ms preprocess, 13.0ms inference, 0.0ms loss, 1.0ms postprocess per image
{'model': 'R5_cdl_gated_age_aug', 'precision': 0.35935625638567736, 'recall': 0.1865554344600992, 'mAP50': 0.1556465236389497, 'mAP50-95': 0.0646960564823801, 'AP50_D00': 0.18531833424834482, 'AP50_D10': 0.13606478135480676, 'AP50_D20': 0.19528962035740718, 'AP50_D40': 0.10591335859524006, 'params_M': 3.782, 'test_set': 'test_dark_mild'}

========== R5_cdl_gated_age_aug | test_dark_severe ==========

Running evaluation: R5_cdl_gated_age_aug | test_dark_severe
Ultralytics 8.3.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
cdl_age_r5 summary: 192 layers, 3,782,117 parameters, 0 gradients
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3004.9±2311.8 MB/s, size: 601.5 KB)



val: Scanning /kaggle/working/rdd2022_reproduction/test_dark_severe/labels.cache... 403 images, 0 backgrounds, 0 corrupt: 100%|██████████| 403/403 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 26/26 [00:07<00:00,  3.39it/s]

                   all        403        955       0.11     0.0651     0.0331     0.0127
Speed: 0.2ms preprocess, 13.0ms inference, 0.0ms loss, 1.0ms postprocess per image
{'model': 'R5_cdl_gated_age_aug', 'precision': 0.10957879246456498, 'recall': 0.06510839566361398, 'mAP50': 0.033072899965172556, 'mAP50-95': 0.012749874202732474, 'AP50_D00': 0.04404894856826829, 'AP50_D10': 0.02945956552904922, 'AP50_D20': 0.05464019975446713, 'AP50_D40': 0.0041428860089055795, 'params_M': 3.782, 'test_set': 'test_dark_severe'}

REPRODUCED RESULTS


,model,precision,recall,mAP50,mAP50-95,AP50_D00,AP50_D10,AP50_D20,AP50_D40,params_M,test_set
0,R1_baseline_scratch,0.424075,0.309351,0.274339,0.115337,0.305619,0.228883,0.404014,0.158842,2.583,test_clean
1,R1_baseline_scratch,0.233022,0.164858,0.112483,0.044561,0.175073,0.104667,0.130918,0.039273,2.583,test_dark_mild
2,R1_baseline_scratch,0.050203,0.031620,0.008478,0.003091,0.017957,0.010073,0.005420,0.000463,2.583,test_dark_severe
3,R2_cdl_scratch,0.368558,0.365463,0.300898,0.137993,0.333691,0.273387,0.423076,0.173440,3.618,test_clean
4,R2_cdl_scratch,0.231785,0.177195,0.108558,0.046052,0.159972,0.098710,0.119486,0.056064,3.618,test_dark_mild
5,R2_cdl_scratch,0.049212,0.044103,0.012537,0.004687,0.023699,0.013290,0.012521,0.000637,3.618,test_dark_severe
6,R3_cdl_aug,0.343591,0.350844,0.294297,0.129383,0.312314,0.259991,0.431465,0.173417,3.618,test_clean
7,R3_cdl_aug,0.251658,0.216285,0.144173,0.062214,0.181403,0.130426,0.205333,0.059532,3.618,test_dark_mild
8,R3_cdl_aug,0.088298,0.059389,0.028305,0.010306,0.040251,0.032428,0.036856,0.003683,3.618,test_dark_severe
9,R4_cdl_age_aug,0.373265,0.326049,0.274602,0.118816,0.297803,0.241326,0.400328,0.158951,3.782,test_clean


## 7. Main reproduced results

In [15]:

main_reproduced = reproduced_df.pivot(
    index="model",
    columns="test_set",
    values="mAP50"
).reindex(index=model_order, columns=test_order)

main_reproduced


test_set,test_clean,test_dark_mild,test_dark_severe
model,,,
R1_baseline_scratch,0.274339,0.112483,0.008478
R2_cdl_scratch,0.300898,0.108558,0.012537
R3_cdl_aug,0.294297,0.144173,0.028305
R4_cdl_age_aug,0.274602,0.144601,0.027995
R5_cdl_gated_age_aug,0.301798,0.155647,0.033073


In [16]:

# Human-readable summary table.
summary = main_reproduced.copy()
summary.columns = ["Clean mAP50", "Mild-dark mAP50", "Severe-dark mAP50"]
summary


,Clean mAP50,Mild-dark mAP50,Severe-dark mAP50
model,,,
R1_baseline_scratch,0.274339,0.112483,0.008478
R2_cdl_scratch,0.300898,0.108558,0.012537
R3_cdl_aug,0.294297,0.144173,0.028305
R4_cdl_age_aug,0.274602,0.144601,0.027995
R5_cdl_gated_age_aug,0.301798,0.155647,0.033073



## 8. Validate against the saved reported results

The repository contains `main_results_table.csv` / `results_tables.csv` generated by the original experiment.

The next cell reads the saved results and compares the reported mAP50 values with the values obtained from the committed checkpoints.

Small numerical differences can occur because inference/runtime versions or hardware can affect evaluation details. A large discrepancy should be investigated rather than silently accepted.


In [17]:

import pandas as pd

SAVED_MAIN = REPO_ROOT / "main_results_table.csv"
SAVED_RESULTS = REPO_ROOT / "results_tables.csv"

saved_path = SAVED_MAIN if SAVED_MAIN.exists() else SAVED_RESULTS

if not saved_path.exists():
    raise FileNotFoundError("Neither saved results CSV was found.")

saved_df = pd.read_csv(saved_path)

print("Comparing against:", saved_path)
print("Saved model names:", sorted(saved_df["model"].astype(str).unique()))

# Convert categorical columns to ordinary strings first.
reproduced_compare = reproduced_df[
    ["model", "test_set", "mAP50"]
].copy()

reproduced_compare["model"] = (
    reproduced_compare["model"].astype(str).str.strip()
)
reproduced_compare["test_set"] = (
    reproduced_compare["test_set"].astype(str).str.strip()
)

saved_compare = saved_df[
    ["model", "test_set", "mAP50"]
].copy()

saved_compare["model"] = saved_compare["model"].astype(str).str.strip()
saved_compare["test_set"] = saved_compare["test_set"].astype(str).str.strip()

# Translate reproduced names into the saved CSV's naming convention.
model_aliases = {
    "R1_baseline_scratch": "R1_baseline",
    "R2_cdl_scratch": "R2_cdl",
    "R3_cdl_aug": "R3_cdl_aug",
    "R4_cdl_age_aug": "R4_cdl_age_aug",
    "R5_cdl_gated_age_aug": "R5_cdl_gated_age_aug",
}

reproduced_compare["model"] = (
    reproduced_compare["model"].replace(model_aliases)
)

comparison = reproduced_compare.merge(
    saved_compare,
    on=["model", "test_set"],
    how="left",
    suffixes=("_reproduced", "_saved"),
    validate="one_to_one",
)

comparison["absolute_difference"] = (
    comparison["mAP50_reproduced"] - comparison["mAP50_saved"]
).abs()

# Restore the original experiment names for display.
reverse_aliases = {v: k for k, v in model_aliases.items()}
comparison["model"] = comparison["model"].replace(reverse_aliases)

display(comparison)


Comparing against: /kaggle/input/datasets/irineannajohnson/rdd22022-project/rdd2022_project/main_results_table.csv
Saved model names: ['R1_baseline', 'R2_cdl', 'R3_cdl_aug', 'R4_cdl_age_aug', 'R5_cdl_gated_age_aug']


,model,test_set,mAP50_reproduced,mAP50_saved,absolute_difference
0,R1_baseline_scratch,test_clean,0.274339,0.274339,0.000000e+00
1,R1_baseline_scratch,test_dark_mild,0.112483,0.112483,4.163336e-17
2,R1_baseline_scratch,test_dark_severe,0.008478,0.008478,9.194034e-17
3,R2_cdl_scratch,test_clean,0.300898,0.300898,0.000000e+00
4,R2_cdl_scratch,test_dark_mild,0.108558,0.108558,8.326673e-17
5,R2_cdl_scratch,test_dark_severe,0.012537,0.012537,2.949030e-17
6,R3_cdl_aug,test_clean,0.294297,0.294297,0.000000e+00
7,R3_cdl_aug,test_dark_mild,0.144173,0.144173,0.000000e+00
8,R3_cdl_aug,test_dark_severe,0.028305,0.028305,2.081668e-17
9,R4_cdl_age_aug,test_clean,0.274602,0.274602,5.551115e-17


In [18]:

# Highlight any discrepancy larger than a small tolerance.
TOLERANCE = 0.005

if comparison["mAP50_saved"].isna().any():
    print("WARNING: Some saved rows were not found in the repository CSV.")
else:
    bad = comparison[comparison["absolute_difference"] > TOLERANCE]
    if len(bad) == 0:
        print(f"All reproduced mAP50 values are within ±{TOLERANCE} of the saved table.")
    else:
        print(f"WARNING: {len(bad)} value(s) differ by more than ±{TOLERANCE}:")
        display(bad)


All reproduced mAP50 values are within ±0.005 of the saved table.


## 9. Per-class AP50

In [19]:

per_class = reproduced_df.pivot_table(
    index=["model", "test_set"],
    values=["AP50_D00", "AP50_D10", "AP50_D20", "AP50_D40"],
    aggfunc="first",
).reindex(
    pd.MultiIndex.from_product(
        [model_order, test_order],
        names=["model", "test_set"]
    )
)

per_class


/tmp/ipykernel_551/3506994694.py:1: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  per_class = reproduced_df.pivot_table(


AP50_D00  AP50_D10  AP50_D20  AP50_D40
model                test_set                                                
R1_baseline_scratch  test_clean        0.305619  0.228883  0.404014  0.158842
                     test_dark_mild    0.175073  0.104667  0.130918  0.039273
                     test_dark_severe  0.017957  0.010073  0.005420  0.000463
R2_cdl_scratch       test_clean        0.333691  0.273387  0.423076  0.173440
                     test_dark_mild    0.159972  0.098710  0.119486  0.056064
                     test_dark_severe  0.023699  0.013290  0.012521  0.000637
R3_cdl_aug           test_clean        0.312314  0.259991  0.431465  0.173417
                     test_dark_mild    0.181403  0.130426  0.205333  0.059532
                     test_dark_severe  0.040251  0.032428  0.036856  0.003683
R4_cdl_age_aug       test_clean        0.297803  0.241326  0.400328  0.158951
                     test_dark_mild    0.172514  0.118037  0.208620  0.079233
                     test_dark_severe  0.046346  0.014691  0.048613  0.002329
R5_cdl_gated_age_aug test_clean        0.325949  0.279549  0.442316  0.159378
                     test_dark_mild    0.185318  0.136065  0.195290  0.105913
                     test_dark_severe  0.044049  0.029460  0.054640  0.004143

## 10. Degradation from clean to dark

In [20]:

pivot = reproduced_df.pivot_table(
    index="model",
    columns="test_set",
    values="mAP50",
    aggfunc="first",
).reindex(index=model_order)

degradation = pd.DataFrame({
    "test_clean": pivot["test_clean"],
    "drop_to_mild": pivot["test_clean"] - pivot["test_dark_mild"],
    "drop_to_severe": pivot["test_clean"] - pivot["test_dark_severe"],
})

degradation


/tmp/ipykernel_551/1188056080.py:1: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = reproduced_df.pivot_table(


,test_clean,drop_to_mild,drop_to_severe
model,,,
R1_baseline_scratch,0.274339,0.161856,0.265861
R2_cdl_scratch,0.300898,0.192340,0.288362
R3_cdl_aug,0.294297,0.150123,0.265992
R4_cdl_age_aug,0.274602,0.130001,0.246607
R5_cdl_gated_age_aug,0.301798,0.146152,0.268725



## 11. Optional: verify the saved project result tables

The following cells do **not** rerun the experiment. They simply inspect the CSV artifacts already committed to the repository.


In [21]:

for filename in [
    "main_results_table.csv",
    "per_class_ap_table.csv",
    "degradation_table.csv",
    "per_country_map_R2_vs_R5.csv",
    "small_defect_subset_table.csv",
]:
    p = REPO_ROOT / filename
    if p.exists():
        print(f"\n### {filename}")
        display(pd.read_csv(p).head(20))
    else:
        print("Missing:", filename)



### main_results_table.csv


,model,test_set,precision,recall,mAP50,mAP50-95,AP50_D00,AP50_D10,AP50_D20,AP50_D40,params_M,GFLOPs,FPS
0,R1_baseline,test,0.424075,0.309351,0.274339,0.115337,0.305619,0.228883,0.404014,0.158842,2.583,NaN,98.6
1,R2_cdl,test,0.368558,0.365463,0.300898,0.137993,0.333691,0.273387,0.423076,0.173440,3.618,NaN,34.5
2,R1_baseline,test_clean,0.424075,0.309351,0.274339,0.115337,0.305619,0.228883,0.404014,0.158842,2.583,NaN,98.6
3,R1_baseline,test_dark_mild,0.233022,0.164858,0.112483,0.044561,0.175073,0.104667,0.130918,0.039273,2.583,NaN,98.6
4,R1_baseline,test_dark_severe,0.050203,0.031620,0.008478,0.003091,0.017957,0.010073,0.005420,0.000463,2.583,NaN,98.6
5,R2_cdl,test_clean,0.368558,0.365463,0.300898,0.137993,0.333691,0.273387,0.423076,0.173440,3.618,NaN,34.5
6,R2_cdl,test_dark_mild,0.231785,0.177195,0.108558,0.046052,0.159972,0.098710,0.119486,0.056064,3.618,NaN,34.5
7,R2_cdl,test_dark_severe,0.049212,0.044103,0.012537,0.004687,0.023699,0.013290,0.012521,0.000637,3.618,NaN,34.5
8,R3_cdl_aug,test_clean,0.343591,0.350844,0.294297,0.129383,0.312314,0.259991,0.431465,0.173417,3.618,NaN,65.5
9,R3_cdl_aug,test_dark_mild,0.251658,0.216285,0.144173,0.062214,0.181403,0.130426,0.205333,0.059532,3.618,NaN,65.5



### per_class_ap_table.csv


,model,test_set,AP50_D00,AP50_D10,AP50_D20,AP50_D40
0,R1_baseline,test_clean,0.305619,0.228883,0.404014,0.158842
1,R1_baseline,test_dark_mild,0.175073,0.104667,0.130918,0.039273
2,R1_baseline,test_dark_severe,0.017957,0.010073,0.005420,0.000463
3,R2_cdl,test_clean,0.333691,0.273387,0.423076,0.173440
4,R2_cdl,test_dark_mild,0.159972,0.098710,0.119486,0.056064
5,R2_cdl,test_dark_severe,0.023699,0.013290,0.012521,0.000637
6,R3_cdl_aug,test_clean,0.312314,0.259991,0.431465,0.173417
7,R3_cdl_aug,test_dark_mild,0.181403,0.130426,0.205333,0.059532
8,R3_cdl_aug,test_dark_severe,0.040251,0.032428,0.036856,0.003683
9,R4_cdl_age_aug,test_clean,0.297803,0.241326,0.400328,0.158951



### degradation_table.csv


,model,test_clean,drop_to_mild,drop_to_severe
0,R1_baseline,0.274339,0.161856,0.265861
1,R2_cdl,0.300898,0.192340,0.288362
2,R3_cdl_aug,0.294297,0.150123,0.265992
3,R4_cdl_age_aug,0.274602,0.130001,0.246607
4,R5_cdl_gated_age_aug,0.301798,0.146152,0.268725



### per_country_map_R2_vs_R5.csv


,country,n_images,mAP50_R2,mAP50_R5,delta
0,Japan,133,0.353630,0.363462,0.009832
1,India,55,0.277089,0.255248,-0.021841
2,Czech,18,0.327322,0.563776,0.236455
3,Norway,49,0.214052,0.119065,-0.094987
4,United_States,82,0.336048,0.332292,-0.003756
5,China_MotorBike,33,0.415495,0.389123,-0.026372
6,China_Drone,33,0.341677,0.373530,0.031853



### small_defect_subset_table.csv


,model,recall,mAP50
0,R1_baseline,0.164871,0.200152
1,R2_cdl,0.250971,0.225026
2,R3_cdl_aug,0.217638,0.193461
3,R4_cdl_age_aug,0.204450,0.230163
4,R5_cdl_gated_age_aug,0.206782,0.176622



## 12. Reproduction conclusion

If the checkpoint-loading and evaluation cells run successfully and the reproduced values are consistent with the saved CSV tables, the repository passes the **saved-results reproduction/validation** requirement.

### What this notebook deliberately does NOT do

- It does not download the original RDD2022 release.
- It does not create a new train/validation/test split.
- It does not regenerate the synthetic dark test sets.
- It does not train R1–R5.
- It does not modify the committed checkpoints or experiment artifacts.

For the complete experiment from scratch, use:

`RDD2022_YOLOv11n_CDL+Adaptive_AGE_full_experiment_from_scratch.ipynb`


## Reproduction-only integrity check

This notebook intentionally does not retrain any model. It loads saved checkpoints and uses Ultralytics `model.val(...)` to reproduce the reported metrics on the saved test sets. 

